# eval_fullcorpus — retrieval on R, candidate pool, recall + oracle

Full-corpus retrieval over ~374k trials: BM25 (whole doc) + dense (retriever on the retrieval
representation, `retriever_max_tokens`) → RRF hybrid → per-topic candidate pool. Writes the pool
and the retrieval features (`bm25/dense/rrf`) that `train_ensemble_full` and the judge consume.
Retrieval isn't the binding constraint (oracle ~0.96), so the retriever stays pre-R for now (§2g step 4).


## Setup (Colab — GPU for the dense encode; BM25 is CPU/slow, cached)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q rank-bm25 pytrec_eval sentence-transformers datasets pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json, pickle
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from ctmatch.experiments import (ExperimentConfig, load_corpus, load_eval, build_bm25, encode_corpus,
                                 full_blob, rrf_fuse, ndcg_at_k, recall_at_k)
cfg = ExperimentConfig(data_root=DATA_ROOT)
CAND_K = cfg.cand_k   # 1000
SETS = ['trec21', 'kz', 'trec22']
os.makedirs(cfg.path('cache'), exist_ok=True); os.makedirs(cfg.path('data'), exist_ok=True)
print('repr:', cfg.repr_tag(), '| retriever:', cfg.retriever_ckpt, '@', cfg.retriever_max_tokens)


In [ ]:
corpus_ids, corpus_fields = load_corpus(cfg)
sets = load_eval(cfg, SETS)
topics = {s: [t for t in sets[s]['rel_dict'] if t in sets[s]['topic2text']] for s in SETS}
print(f'{len(corpus_ids):,} docs |', {s: len(v) for s, v in topics.items()}, 'topics')


In [ ]:
# BM25 over the whole document (cached — the index build is the slow part).
bm25_path = cfg.bm25_file('fulltext_R')
if os.path.exists(bm25_path):
    bm25 = pickle.load(open(bm25_path, 'rb'))
else:
    bm25 = build_bm25(corpus_fields, cfg); pickle.dump(bm25, open(bm25_path, 'wb'))
def bm25_topk(qtext, k):
    scores = bm25.get_scores(qtext.lower().split())
    top = np.argpartition(-scores, k)[:k]; top = top[np.argsort(-scores[top])]
    return {corpus_ids[i]: float(scores[i]) for i in top}


In [ ]:
# Dense: encode the corpus once on the retrieval representation (cached), then cosine per topic.
emb_path = cfg.emb_file('fulltext_R')
if os.path.exists(emb_path):
    doc_emb = np.load(emb_path)
else:
    doc_emb = encode_corpus(corpus_fields, cfg); np.save(emb_path, doc_emb)
from sentence_transformers import SentenceTransformer
q_enc = SentenceTransformer(cfg.retriever_ckpt); q_enc.max_seq_length = cfg.retriever_max_tokens
def dense_topk(qtext, k):
    q = q_enc.encode([qtext], normalize_embeddings=True)[0].astype('float32')
    sims = doc_emb @ q
    top = np.argpartition(-sims, k)[:k]; top = top[np.argsort(-sims[top])]
    return {corpus_ids[i]: float(sims[i]) for i in top}


In [ ]:
# Retrieve BM25 + dense per topic; RRF-fuse; candidate pool = union. Cache per-topic feature scores.
pool, feats = {}, {}   # pool[s][t] = [doc_ids]; feats[(s,t,d)] = {bm25,dense,rrf,bm25_rank,dense_rank}
for s in SETS:
    pool[s] = {}
    for t in tqdm(topics[s], desc=f'retrieve {s}'):
        qt = sets[s]['topic2text'][t]
        bm, dn = bm25_topk(qt, CAND_K), dense_topk(qt, CAND_K)
        bm_rank = {d: r for r, d in enumerate(sorted(bm, key=bm.get, reverse=True))}
        dn_rank = {d: r for r, d in enumerate(sorted(dn, key=dn.get, reverse=True))}
        rrf = rrf_fuse([list(bm_rank), list(dn_rank)], k=cfg.rrf_k)
        cand = sorted(set(bm) | set(dn), key=lambda d: rrf.get(d, 0), reverse=True)
        pool[s][t] = cand
        for d in cand:
            feats[(s, t, d)] = {'bm25': bm.get(d, 0.0), 'dense': dn.get(d, 0.0), 'rrf': rrf.get(d, 0.0),
                                'bm25_rank': bm_rank.get(d, CAND_K), 'dense_rank': dn_rank.get(d, CAND_K)}


In [ ]:
# Retrieval quality: recall@100/1000 (eligible-only) + oracle NDCG@10 on the hybrid pool.
rows = []
for s in SETS:
    rel = sets[s]['rel_dict']
    r100 = np.mean([recall_at_k(pool[s][t], rel[t], 100, rel_level=2) for t in topics[s]])
    r1000 = np.mean([recall_at_k(pool[s][t], rel[t], 1000, rel_level=2) for t in topics[s]])
    oracle = np.mean([ndcg_at_k(sorted(pool[s][t], key=lambda d: rel[t].get(d, 0), reverse=True), rel[t]) for t in topics[s]])
    rows.append({'split': s, 'recall@100': round(float(r100),3), 'recall@1000': round(float(r1000),3),
                 'oracle_ndcg@10': round(float(oracle),3), 'mean_pool': int(np.mean([len(pool[s][t]) for t in topics[s]]))})
pd.DataFrame(rows)


In [ ]:
# Persist pool + retrieval features for the judge and the ensemble.
json.dump({s: pool[s] for s in SETS}, open(cfg.path('data/pool_R.json'), 'w'))
with open(cfg.path('data/retrieval_feats_R.jsonl'), 'w') as f:
    for (s, t, d), v in feats.items():
        f.write(json.dumps({'source': s, 'topic_id': t, 'doc_id': d, **v}) + '\n')
print('wrote data/pool_R.json + data/retrieval_feats_R.jsonl')
